# Descarga de Predictores - Base de Commodities

**Universidad de Buenos Aires - Facultad de Ciencias Económicas**  
**Taller de Programación - Big Data**  
**Año 2025**

---

## Objetivo

Descargar **predictores incrementales** para mejorar los modelos de predicción de precios de commodities. Este notebook está diseñado para ser **escalable** y permitir agregar nuevos predictores de manera ordenada.

### Pipeline de predictores

1. **Indicadores de volatilidad y riesgo** - VIX, MOVE, VXN
2. **Índices macroeconómicos** - DXY (dólar), tasas de interés
3. **Índices sectoriales** - S&P 500, sectores específicos
4. **Variables climáticas** - Temperaturas, precipitaciones (para agrícolas)
5. **Sentiment/alternativas** - Twitter sentiment, búsquedas Google Trends

**Estructura incremental:**
- Cada predictor se descarga, limpia y guarda en `data/interim/predictors/`
- Se mantiene un registro JSON con metadata de cada predictor
- Al finalizar, se consolida todo en `data/processed/predictors_consolidated.csv`

---

## ⚠️ IMPORTANTE

Este notebook **solo descarga predictores**. El procesamiento final y merge con precios de commodities se hace en `03_process_data.ipynb`.


## 0. Configuración Inicial

In [1]:
# Imports
import os
import sys
import warnings
import json
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Estilo de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Librerías cargadas correctamente")
print(f"Pandas versión: {pd.__version__}")
print(f"yfinance versión: {yf.__version__}")

✓ Librerías cargadas correctamente
Pandas versión: 2.2.3
yfinance versión: 0.2.65


In [2]:
# Definir rutas
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / 'data' / 'raw'
INTERIM_DIR = BASE_DIR / 'data' / 'interim'
PREDICTORS_DIR = INTERIM_DIR / 'predictors'  # Nueva carpeta para predictores
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

# Crear carpetas de salida
PREDICTORS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Estructura de directorios verificada")
print(f"  Raw: {RAW_DIR.relative_to(BASE_DIR)}")
print(f"  Interim: {INTERIM_DIR.relative_to(BASE_DIR)}")
print(f"  Predictors: {PREDICTORS_DIR.relative_to(BASE_DIR)}")
print(f"  Processed: {PROCESSED_DIR.relative_to(BASE_DIR)}")

✓ Estructura de directorios verificada
  Raw: data\raw
  Interim: data\interim
  Predictors: data\interim\predictors
  Processed: data\processed


### Registro de predictores (metadata)

Cada predictor descargado se registra en un archivo JSON para mantener trazabilidad.

In [3]:
# Inicializar archivo de registro de predictores
METADATA_PATH = PREDICTORS_DIR / 'predictors_registry.json'

def load_predictors_registry():
    """Cargar registro de predictores existente"""
    if METADATA_PATH.exists():
        with open(METADATA_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    else:
        return {
            'last_updated': None,
            'predictors': {}
        }

def save_predictor_metadata(name, metadata):
    """Guardar metadata de un predictor en el registro"""
    registry = load_predictors_registry()
    
    registry['last_updated'] = datetime.now().isoformat()
    registry['predictors'][name] = metadata
    
    with open(METADATA_PATH, 'w', encoding='utf-8') as f:
        json.dump(registry, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Metadata guardada: {name}")

# Cargar registro actual
registry = load_predictors_registry()

if registry['predictors']:
    print(f"\n📋 Predictores ya descargados: {len(registry['predictors'])}")
    for name in registry['predictors'].keys():
        print(f"  - {name}")
else:
    print("\n📋 No hay predictores descargados aún (registro vacío)")


📋 No hay predictores descargados aún (registro vacío)


---

## 1. Indicadores de Volatilidad y Riesgo

### 1.1. VIX - CBOE Volatility Index

**Ticker:** ^VIX | **Fuente:** Yahoo Finance | **Relación:** ↑VIX → ↓commodities riesgosos, ↑oro

In [ ]:
# ============================================================================
# VIX - DESCARGA, LIMPIEZA, EXPORTACIÓN Y VISUALIZACIÓN
# ============================================================================

print("="*60)
print("VIX - CBOE Volatility Index")
print("="*60)

# 1. DESCARGA
df = yf.download('^VIX', start='2000-01-01', progress=False).reset_index()

# Manejar MultiIndex si existe (yfinance a veces devuelve MultiIndex)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)  # Eliminar segundo nivel del MultiIndex
    
df.columns = [col.lower() if isinstance(col, str) else col for col in df.columns]

if 'date' not in df.columns:
    df = df.rename(columns={'index': 'date'})
    
df['predictor'] = 'VIX'
df = df.sort_values('date').reset_index(drop=True)

print(f"\n✓ Descargado: {len(df):,} obs | {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Media: {df['close'].mean():.2f} | Min: {df['close'].min():.2f} | Max: {df['close'].max():.2f}")

# 2. LIMPIEZA
df['close'] = df['close'].clip(lower=0).ffill().bfill()  # Eliminar negativos + imputar
df = df.drop_duplicates(subset='date', keep='first').reset_index(drop=True)

# 3. EXPORTACIÓN
output_path = PREDICTORS_DIR / 'vix.csv'
df.to_csv(output_path, index=False)
print(f"✓ Exportado: {output_path.relative_to(BASE_DIR)} ({output_path.stat().st_size/1024:.1f} KB)")

# Metadata
save_predictor_metadata('VIX', {
    'ticker': '^VIX', 'name': 'CBOE Volatility Index',
    'description': 'Volatilidad implícita S&P 500 (miedo del mercado)',
    'source': 'Yahoo Finance', 'frequency': 'Daily',
    'period': {'start': df['date'].min().isoformat(), 'end': df['date'].max().isoformat()},
    'observations': len(df), 'file': 'vix.csv',
    'stats': {'mean': float(df['close'].mean()), 'min': float(df['close'].min()), 'max': float(df['close'].max())}
})

# 4. VISUALIZACIÓN
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['date'], df['close'], linewidth=1, color='darkred', alpha=0.8)
ax.fill_between(df['date'], df['close'], alpha=0.2, color='red')
ax.axhline(15, color='green', linestyle='--', alpha=0.4, label='Baja vol (<15)')
ax.axhline(30, color='orange', linestyle='--', alpha=0.4, label='Alta vol (>30)')
ax.set_title('VIX - CBOE Volatility Index (2000-2025)', fontsize=13, fontweight='bold')
ax.set_ylabel('VIX'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()

graficos_dir = BASE_DIR / 'graficos'
graficos_dir.mkdir(exist_ok=True)
plt.savefig(graficos_dir / 'predictor_vix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Gráfico: graficos/predictor_vix.png\n")
df_vix = df  # Guardar para uso posterior

VIX - CBOE Volatility Index


AttributeError: Can only use .str accessor with Index, not MultiIndex

---

## 2. Próximos Predictores (Plantilla)

**Para agregar un nuevo predictor**, copiar y modificar la celda de VIX con estos cambios:
- Ticker de Yahoo Finance (ej: `'DX-Y.NYB'` para Dollar Index, `'^GSPC'` para S&P 500)
- Nombre del archivo CSV (ej: `'dxy.csv'`, `'sp500.csv'`)
- Título y color del gráfico

**Predictores sugeridos:**
- **DXY (Dollar Index):** `DX-Y.NYB` - Relación inversa con commodities
- **S&P 500:** `^GSPC` - Apetito de riesgo general
- **Treasury 10Y:** `^TNX` - Proxy de tasas de interés
- **Índice Energía:** `^GSPE` - Correlación con commodities energéticos
- **Índice Materiales:** `^GSPMS` - Correlación con metales


---

## 3. Resumen Final

In [ ]:
# Resumen final
registry = load_predictors_registry()
print("="*80)
print(f"PREDICTORES DESCARGADOS: {len(registry['predictors'])}")
print("="*80)

for name, meta in registry['predictors'].items():
    print(f"\n{name}: {meta['observations']:,} obs | {meta['period']['start'][:10]} → {meta['period']['end'][:10]}")
    print(f"  Archivo: {meta['file']} | Ticker: {meta['ticker']}")

print(f"\n📁 Archivos en: {PREDICTORS_DIR.relative_to(BASE_DIR)}/")
print("\n💡 Próximo paso: Ejecutar 03_process_data.ipynb para consolidar con commodities")